In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/raw")
visits = pd.read_csv(DATA / "air_visit_data.csv", parse_dates=["visit_date"])
print(visits.shape)
print(visits.dtypes)
visits.info()

(252108, 3)
air_store_id               str
visit_date      datetime64[us]
visitors                 int64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 252108 entries, 0 to 252107
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   air_store_id  252108 non-null  str           
 1   visit_date    252108 non-null  datetime64[us]
 2   visitors      252108 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 5.8 MB


In [7]:
# 1) print() ke saath label, sabse clear
print("Missing values per column:")
print(visits.isna().sum())
print("Duplicate (store, date) rows:", visits.duplicated(["air_store_id", "visit_date"]).sum())
print("Date range:", visits["visit_date"].min(), "to", visits["visit_date"].max())
print("Unique stores:", visits["air_store_id"].nunique())


Missing values per column:
air_store_id    0
visit_date      0
visitors        0
dtype: int64
Duplicate (store, date) rows: 0
Date range: 2016-01-01 00:00:00 to 2017-04-22 00:00:00
Unique stores: 829


In [9]:
# Per-store span: first visit to last visit for each store
g = visits.groupby("air_store_id")["visit_date"].agg(["min", "max", "count"])

# +1 because both the first and last day are counted (inclusive range)
g["span_days"] = (g["max"] - g["min"]).dt.days + 1
g["missing_days"] = g["span_days"] - g["count"]

print(g["missing_days"].describe())
print("Stores with zero missing days:", (g["missing_days"] == 0).sum())

# Overall coverage across the full date range
n_days = (visits["visit_date"].max() - visits["visit_date"].min()).days + 1
n_stores = visits["air_store_id"].nunique()
print("Days in range:", n_days)
print("Max possible rows:", n_stores * n_days)
print("Coverage:", round(len(visits) / (n_stores * n_days), 3))

count    829.000000
mean      53.282268
std       47.504766
min        0.000000
25%       16.000000
50%       47.000000
75%       74.000000
max      351.000000
Name: missing_days, dtype: float64
Stores with zero missing days: 11
Days in range: 478
Max possible rows: 396262
Coverage: 0.636
